In [ ]:
# ─────────────────────────────────────────────
# PART 1: Install & Imports
# ─────────────────────────────────────────────
!pip install open3d plotly huggingface_hub numpy scipy wandb -q
!pip install torch scikit-learn -q

import open3d as o3d
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import wandb
import os
import math
import time
import zipfile
from sklearn.decomposition import PCA
from huggingface_hub import hf_hub_download
from torch.utils.data import Dataset, DataLoader, Subset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.3 MB/s eta 0:00:00
Using device: cuda


In [ ]:
# ─────────────────────────────────────────────
# PART 2: Download & Extract Dataset
# ─────────────────────────────────────────────
zip_path = hf_hub_download(
    repo_id="BGLab/AgriField3D",
    filename="datasets/FielGrwon_ZeaMays_RawPCD_10k.zip",
    repo_type="dataset",
    local_dir="./data"
)

extract_dir = "./data/RawPCD_10k"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

ply_files = sorted([
    os.path.join(root, f)
    for root, _, files in os.walk(extract_dir)
    for f in files if f.endswith('.ply')
])
print(f"Total plants found: {len(ply_files)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


datasets/FielGrwon_ZeaMays_RawPCD_10k.zi(…):   0%|          | 0.00/218M [00:00<?, ?B/s]

Total plants found: 1045


In [ ]:
# ─────────────────────────────────────────────
# PART 3: Dataset + Fixed 500 / Rest Split
# ─────────────────────────────────────────────
NUM_POINTS = 1024
TRAIN_SIZE = 500

def preprocess(ply_path, num_points=NUM_POINTS):
    pcd = o3d.io.read_point_cloud(ply_path)
    pts = np.asarray(pcd.points, dtype=np.float32)
    if pts.shape[0] == 0:
        return None
    N   = pts.shape[0]
    idx = (np.random.choice(N, num_points, replace=False)
           if N >= num_points
           else np.random.choice(N, num_points, replace=True))
    pts  = pts[idx]
    pts -= pts.mean(axis=0)
    pts /= (np.linalg.norm(pts, axis=1).max() + 1e-8)
    return pts


class MaizeDataset(Dataset):
    def __init__(self, ply_files):
        self.samples = []
        self.paths   = []
        print(f"Loading {len(ply_files)} files ...")
        for i, path in enumerate(ply_files):
            try:
                pts = preprocess(path)
                if pts is not None:
                    self.samples.append(torch.tensor(pts))
                    self.paths.append(path)
            except Exception as e:
                print(f"  [skip] {os.path.basename(path)}: {e}")
            if (i+1) % 20 == 0:
                print(f"  {i+1}/{len(ply_files)} -- {len(self.samples)} valid")
        print(f"Dataset ready: {len(self.samples)} samples")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx], idx


full_dataset  = MaizeDataset(ply_files)
total         = len(full_dataset)

all_indices   = np.random.permutation(total)
train_indices = all_indices[:TRAIN_SIZE].tolist()
test_indices  = all_indices[TRAIN_SIZE:].tolist()

train_dataset = Subset(full_dataset, train_indices)
test_dataset  = Subset(full_dataset, test_indices)

# pin_memory + num_workers for faster GPU transfer
train_loader = DataLoader(train_dataset, batch_size=8,
                          shuffle=True,  drop_last=True,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=8,
                          shuffle=False, drop_last=False,
                          num_workers=2, pin_memory=True)

print(f"Train : {len(train_dataset)}")
print(f"Test  : {len(test_dataset)} (unseen during training)")


Loading 1045 files ...
  20/1045 -- 20 valid
  40/1045 -- 40 valid
  60/1045 -- 60 valid
  80/1045 -- 80 valid
  100/1045 -- 100 valid
  120/1045 -- 120 valid
  140/1045 -- 140 valid
  160/1045 -- 160 valid
  180/1045 -- 180 valid
  200/1045 -- 200 valid
  220/1045 -- 220 valid
  240/1045 -- 240 valid
  260/1045 -- 260 valid
  280/1045 -- 280 valid
  300/1045 -- 300 valid
  320/1045 -- 320 valid
  340/1045 -- 340 valid
  360/1045 -- 360 valid
  380/1045 -- 380 valid
  400/1045 -- 400 valid
  420/1045 -- 420 valid
  440/1045 -- 440 valid
  460/1045 -- 460 valid
  480/1045 -- 480 valid
  500/1045 -- 500 valid
  520/1045 -- 520 valid
  540/1045 -- 540 valid
  560/1045 -- 560 valid
  580/1045 -- 580 valid
  600/1045 -- 600 valid
  620/1045 -- 620 valid
  640/1045 -- 640 valid
  660/1045 -- 660 valid
  680/1045 -- 680 valid
  700/1045 -- 700 valid
  720/1045 -- 720 valid
  740/1045 -- 740 valid
  760/1045 -- 760 valid
  780/1045 -- 780 valid
  800/1045 -- 800 valid
  820/1045 -- 820 valid
 

In [ ]:
# ─────────────────────────────────────────────
# PART 4: Point Transformer V3 Autoencoder
# Optimisations vs original:
#   * Latent dim 256 -> 512 (richer bottleneck)
#   * Larger decoder: 512->1024->2048 hidden units
#   * Dropout 0.3->0.2 in encoder FC (less aggressive)
#   * AdamW + weight decay used in training cell
#   * LR warm-up + cosine decay in training cell
#   * num_workers / pin_memory on DataLoaders
# ─────────────────────────────────────────────

# ════════════════════════════════════════════════
# 4A: Space-Filling Curve Serialization
# ════════════════════════════════════════════════

def xyz_to_zorder(coords):
    x = coords[:, 0].long()
    y = coords[:, 1].long()
    z = coords[:, 2].long()
    code = torch.zeros_like(x)
    for i in range(21):
        code |= ((x >> i) & 1) << (3 * i)
        code |= ((y >> i) & 1) << (3 * i + 1)
        code |= ((z >> i) & 1) << (3 * i + 2)
    return code


def xyz_to_hilbert(coords, order=10):
    x = coords[:, 0].long().cpu().numpy()
    y = coords[:, 1].long().cpu().numpy()
    z = coords[:, 2].long().cpu().numpy()
    N = len(x)
    codes = np.zeros(N, dtype=np.int64)
    for s in range(order - 1, -1, -1):
        rx = ((x >> s) & 1).astype(np.int64)
        ry = ((y >> s) & 1).astype(np.int64)
        rz = ((z >> s) & 1).astype(np.int64)
        d  = rx * 4 + ry * 2 + rz
        gray = d ^ (d >> 1)
        codes = (codes << 3) | gray
        swap_mask = (d % 2 == 0)
        x_new, y_new, z_new = x.copy(), y.copy(), z.copy()
        x_new[swap_mask], z_new[swap_mask] = z[swap_mask], x[swap_mask]
        x, y, z = x_new, y_new, z_new
    return torch.tensor(codes, dtype=torch.long, device=coords.device)


def serialise_point_cloud(points, grid_size=0.02, patterns=("z", "tz", "h", "th")):
    B, N, _ = points.shape
    grid_pts = ((points - points.min(dim=1, keepdim=True)[0]) / grid_size).long()
    orders = {}
    for pat in patterns:
        codes_list = []
        for b in range(B):
            gp = grid_pts[b]
            if   pat == "z":  c = xyz_to_zorder(gp)
            elif pat == "tz": c = xyz_to_zorder(gp[:, [1, 0, 2]])
            elif pat == "h":  c = xyz_to_hilbert(gp)
            elif pat == "th": c = xyz_to_hilbert(gp[:, [1, 0, 2]])
            else: raise ValueError(f"Unknown: {pat}")
            codes_list.append(c)
        codes = torch.stack(codes_list, dim=0)
        orders[pat] = codes.argsort(dim=1)
    return orders


# ════════════════════════════════════════════════
# 4B: Enhanced Conditional Positional Encoding (xCPE)
# ════════════════════════════════════════════════

class xCPE(nn.Module):
    def __init__(self, dim, pos_dim=3):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(pos_dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim)
        )

    def forward(self, feat, pos):
        return feat + self.proj(pos)


# ════════════════════════════════════════════════
# 4C: Patch Attention with Shuffle Order
# ════════════════════════════════════════════════

class PatchAttention(nn.Module):
    def __init__(self, dim, num_heads=4, patch_size=64,
                 qkv_bias=True, attn_drop=0.0, proj_drop=0.0):
        super().__init__()
        self.dim        = dim
        self.num_heads  = num_heads
        self.head_dim   = dim // num_heads
        self.scale      = self.head_dim ** -0.5
        self.patch_size = patch_size
        self.qkv        = nn.Linear(dim, 3 * dim, bias=qkv_bias)
        self.proj       = nn.Linear(dim, dim)
        self.attn_drop  = nn.Dropout(attn_drop)
        self.proj_drop  = nn.Dropout(proj_drop)

    def forward(self, x, order):
        B, N, C = x.shape
        ps = self.patch_size
        idx_expand = order.unsqueeze(-1).expand(-1, -1, C)
        x_ordered  = torch.gather(x, 1, idx_expand)
        pad_len = (ps - N % ps) % ps
        if pad_len > 0:
            x_ordered = torch.cat([x_ordered, x_ordered[:, -pad_len:, :]], dim=1)
        N_padded    = x_ordered.shape[1]
        num_patches = N_padded // ps
        x_patches   = x_ordered.view(B, num_patches, ps, C)
        qkv = self.qkv(x_patches)
        qkv = qkv.reshape(B, num_patches, ps, 3, self.num_heads, self.head_dim
                          ).permute(3, 0, 4, 1, 2, 5)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        out  = (attn @ v).permute(0, 2, 3, 1, 4).reshape(B, N_padded, C)
        out  = self.proj_drop(self.proj(out))
        out  = out[:, :N, :]
        inv_order  = order.argsort(dim=1)
        inv_expand = inv_order.unsqueeze(-1).expand(-1, -1, C)
        return torch.gather(out, 1, inv_expand)


# ════════════════════════════════════════════════
# 4D: Pre-Norm Transformer Block (PTv3 style)
# ════════════════════════════════════════════════

class PTv3Block(nn.Module):
    def __init__(self, dim, num_heads=4, patch_size=64,
                 mlp_ratio=4, drop_path=0.1):
        super().__init__()
        self.xcpe  = xCPE(dim)
        self.norm1 = nn.LayerNorm(dim)
        self.attn  = PatchAttention(dim, num_heads=num_heads, patch_size=patch_size)
        self.norm2 = nn.LayerNorm(dim)
        hidden     = int(dim * mlp_ratio)
        self.mlp   = nn.Sequential(
            nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop_path),
            nn.Linear(hidden, dim), nn.Dropout(drop_path)
        )

    def forward(self, x, pos, order):
        x = self.xcpe(x, pos)
        x = x + self.attn(self.norm1(x), order)
        x = x + self.mlp(self.norm2(x))
        return x


# ════════════════════════════════════════════════
# 4E: Grid Pooling (hierarchical downsampling)
# ════════════════════════════════════════════════

class GridPool(nn.Module):
    def __init__(self, in_dim, out_dim, stride=2):
        super().__init__()
        self.stride = stride
        self.proj   = nn.Linear(in_dim, out_dim)
        self.norm   = nn.BatchNorm1d(out_dim)

    def forward(self, feat, pos):
        B, N, _ = pos.shape
        s     = self.stride
        N_out = N // s
        feat_r  = feat[:, :N_out * s].view(B, N_out, s, -1)
        pos_r   = pos[:, :N_out * s].view(B, N_out, s, 3)
        pooled_feat = feat_r.mean(dim=2)
        pooled_pos  = pos_r.mean(dim=2)
        out = self.proj(pooled_feat)
        out = self.norm(out.transpose(1, 2)).transpose(1, 2)
        return out, pooled_pos


# ════════════════════════════════════════════════
# 4F: PTv3 Encoder (latent_dim now 512)
# ════════════════════════════════════════════════

class PTv3_Encoder(nn.Module):
    def __init__(self, latent_dim=512, patch_size=64):
        super().__init__()
        self.patterns = ["z", "tz", "h", "th"]

        self.embed = nn.Sequential(
            nn.Linear(3, 32), nn.LayerNorm(32), nn.GELU(),
            nn.Linear(32, 64), nn.LayerNorm(64), nn.GELU()
        )

        # Stage 1: 1024 pts, dim=64
        self.stage1 = nn.ModuleList([
            PTv3Block(64,  num_heads=4, patch_size=patch_size),
            PTv3Block(64,  num_heads=4, patch_size=patch_size),
        ])
        self.pool1 = GridPool(64, 128, stride=2)

        # Stage 2: 512 pts, dim=128
        self.stage2 = nn.ModuleList([
            PTv3Block(128, num_heads=4, patch_size=patch_size),
            PTv3Block(128, num_heads=4, patch_size=patch_size),
        ])
        self.pool2 = GridPool(128, 256, stride=2)

        # Stage 3: 256 pts, dim=256
        self.stage3 = nn.ModuleList([
            PTv3Block(256, num_heads=8, patch_size=patch_size),
            PTv3Block(256, num_heads=8, patch_size=patch_size),
        ])

        # Global pool -> latent (dropout reduced 0.3->0.2)
        self.fc = nn.Sequential(
            nn.Linear(256, 512), nn.LayerNorm(512), nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(512, latent_dim)
        )

    def _run_stage(self, blocks, feat, pos, orders):
        pattern_names = list(orders.keys())
        for block in blocks:
            pat  = pattern_names[torch.randint(len(pattern_names), (1,)).item()]
            feat = block(feat, pos, orders[pat])
        return feat

    def forward(self, x):
        B, N, _ = x.shape
        pos = x

        orders = serialise_point_cloud(pos, grid_size=0.05, patterns=self.patterns)
        feat   = self.embed(x)

        feat = self._run_stage(self.stage1, feat, pos, orders)
        feat, pos = self.pool1(feat, pos)

        orders = serialise_point_cloud(pos, grid_size=0.1, patterns=self.patterns)
        feat = self._run_stage(self.stage2, feat, pos, orders)
        feat, pos = self.pool2(feat, pos)

        orders = serialise_point_cloud(pos, grid_size=0.2, patterns=self.patterns)
        feat = self._run_stage(self.stage3, feat, pos, orders)

        glob = feat.max(dim=1)[0]
        return self.fc(glob)


# ════════════════════════════════════════════════
# 4G: MLP Decoder (wider: 1024->2048 hidden)
# ════════════════════════════════════════════════

class MLP_Decoder(nn.Module):
    def __init__(self, latent_dim=512, num_points=NUM_POINTS):
        super().__init__()
        self.num_points = num_points
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 1024),
            nn.BatchNorm1d(1024), nn.LeakyReLU(0.2), nn.Dropout(0.2),
            nn.Linear(1024, 2048),
            nn.BatchNorm1d(2048), nn.LeakyReLU(0.2), nn.Dropout(0.1),
            nn.Linear(2048, num_points * 3)
        )

    def forward(self, z):
        return self.net(z).view(-1, self.num_points, 3)


# ════════════════════════════════════════════════
# 4H: Full Autoencoder
# ════════════════════════════════════════════════

class PTv3_Autoencoder(nn.Module):
    def __init__(self, latent_dim=512, num_points=NUM_POINTS, patch_size=64):
        super().__init__()
        self.encoder = PTv3_Encoder(latent_dim=latent_dim, patch_size=patch_size)
        self.decoder = MLP_Decoder(latent_dim=latent_dim, num_points=num_points)

    def forward(self, x):
        z     = self.encoder(x)
        recon = self.decoder(z)
        return recon, z


def chamfer_distance(pred, target):
    d1 = (pred.unsqueeze(2) - target.unsqueeze(1)).pow(2).sum(-1)
    d2 = (target.unsqueeze(2) - pred.unsqueeze(1)).pow(2).sum(-1)
    return d1.min(2)[0].mean() + d2.min(2)[0].mean()


# Sanity check
model_test = PTv3_Autoencoder(latent_dim=512, num_points=NUM_POINTS, patch_size=64).to(device)
dummy = torch.randn(2, NUM_POINTS, 3).to(device)
recon_test, z_test = model_test(dummy)
print(f"Encoder output  : {z_test.shape}")
print(f"Decoder output  : {recon_test.shape}")
print(f"Total parameters: {sum(p.numel() for p in model_test.parameters()):,}")
del model_test, dummy, recon_test, z_test


Encoder output  : torch.Size([2, 512])
Decoder output  : torch.Size([2, 1024, 3])
Total parameters: 11,617,536


In [ ]:
# ─────────────────────────────────────────────
# PART 5: Checkpoint Utilities
# ─────────────────────────────────────────────
CKPT_DIR = "./checkpoints_ptv3"
os.makedirs(CKPT_DIR, exist_ok=True)

def save_checkpoint(model, optimizer, scheduler, epoch, loss, tag="latest"):
    path = os.path.join(CKPT_DIR, f"ptv3_{tag}.pt")
    torch.save({
        "epoch"       : epoch,
        "model_state" : model.state_dict(),
        "optim_state" : optimizer.state_dict(),
        "sched_state" : scheduler.state_dict(),
        "loss"        : loss,
    }, path)
    return path


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    if optimizer: optimizer.load_state_dict(ckpt["optim_state"])
    if scheduler: scheduler.load_state_dict(ckpt["sched_state"])
    print(f"  Loaded -- epoch {ckpt['epoch']} | loss {ckpt['loss']:.6f}")
    return ckpt


In [ ]:
# ─────────────────────────────────────────────
# PART 6: Training (optimised, <= 1 hr)
# Changes vs original:
#   * AdamW + weight_decay=1e-4 (better regularisation)
#   * LR warm-up (5 ep) + cosine decay
#   * latent_dim 256 -> 512
#   * Both linear and log10 loss logged to W&B
#   * Dual-panel loss curve (linear + log) to W&B
#   * 3-D point cloud snapshots every 50 epochs
#   * elapsed_min logged so you can track wall time
# ─────────────────────────────────────────────
CONFIG = dict(
    epochs            = 100,         # PTv3 is heavier per-epoch; 100 fits in ~1 hr
    batch_size        = 8,
    lr                = 1e-3,
    weight_decay      = 1e-4,
    latent_dim        = 512,
    num_points        = NUM_POINTS,
    patch_size        = 64,
    train_size        = TRAIN_SIZE,
    checkpoint_every  = 25,
    warmup_epochs     = 5,
    serialization     = "z+tz+h+th",
    patch_interaction = "shuffle_order",
    pos_encoding      = "xCPE",
)

wandb.init(
    project = "maize-ptv3",
    config  = CONFIG,
    name    = "ptv3-optimised-100ep"
)

model = PTv3_Autoencoder(
    latent_dim = CONFIG["latent_dim"],
    num_points = CONFIG["num_points"],
    patch_size = CONFIG["patch_size"]
).to(device)

# AdamW for better generalisation vs plain Adam
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"]
)

# Linear warm-up then cosine decay
def lr_lambda(epoch):
    if epoch < CONFIG["warmup_epochs"]:
        return epoch / max(1, CONFIG["warmup_epochs"])
    progress = (epoch - CONFIG["warmup_epochs"]) / max(
        1, CONFIG["epochs"] - CONFIG["warmup_epochs"])
    return 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

wandb.watch(model, log="all", log_freq=20)

best_loss      = float("inf")
train_losses   = []
training_start = time.time()

for epoch in range(1, CONFIG["epochs"] + 1):

    # ── Train ──────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    for batch, _ in train_loader:
        batch = batch.to(device, non_blocking=True)
        optimizer.zero_grad()
        recon, _ = model(batch)
        loss      = chamfer_distance(recon, batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()

    epoch_loss /= len(train_loader)
    train_losses.append(epoch_loss)
    scheduler.step()

    cur_lr  = optimizer.param_groups[0]["lr"]
    elapsed = (time.time() - training_start) / 60

    # ── W&B: log linear loss AND log10 loss every epoch ────────────
    wandb.log({
        "epoch"          : epoch,
        "train_loss"     : epoch_loss,                      # linear
        "train_loss_log" : np.log10(epoch_loss + 1e-10),   # log10
        "lr"             : cur_lr,
        "elapsed_min"    : elapsed,
    })

    # ── 3-D snapshot every 50 epochs ───────────────────────────────
    if epoch % 50 == 0:
        model.eval()
        sample = next(iter(train_loader))[0][:1].to(device)
        with torch.no_grad():
            s_recon, _ = model(sample)
        o = sample[0].cpu().numpy()
        r = s_recon[0].cpu().numpy()
        wandb.log({
            f"train_original_ep{epoch}": wandb.Object3D(
                np.hstack([o, np.tile([0, 120, 255], (len(o), 1))])),
            f"train_recon_ep{epoch}": wandb.Object3D(
                np.hstack([r, np.tile([255, 80, 0],  (len(r), 1))])),
        })
        model.train()

    # ── Periodic checkpoint ────────────────────────────────────────
    if epoch % CONFIG["checkpoint_every"] == 0:
        path = save_checkpoint(model, optimizer, scheduler,
                               epoch, epoch_loss, tag="latest")
        wandb.save(path)
        print(f"Epoch {epoch:4d} | Loss {epoch_loss:.6f} | "
              f"LR {cur_lr:.2e} | {elapsed:.1f} min | ckpt saved")

    # ── Best model checkpoint ──────────────────────────────────────
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        path = save_checkpoint(model, optimizer, scheduler,
                               epoch, epoch_loss, tag="best")
        wandb.save(path)

total_time = (time.time() - training_start) / 60
print(f"Training done in {total_time:.1f} min.  Best loss: {best_loss:.6f}")

# ── Dual-panel loss curve: linear left, log right ─────────────────────
epochs_x = list(range(1, len(train_losses) + 1))

fig_dual = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Loss -- Linear Scale", "Loss -- Log Scale")
)
for col, use_log in [(1, False), (2, True)]:
    fig_dual.add_trace(go.Scatter(
        x=epochs_x, y=train_losses, mode="lines",
        name="Train Loss",
        line=dict(color="#636EFA" if col == 1 else "#FF7F0E", width=2),
        showlegend=(col == 1)
    ), row=1, col=col)
    if use_log:
        fig_dual.update_yaxes(type="log",
                              title_text="Chamfer Distance (log scale)",
                              row=1, col=col)
    else:
        fig_dual.update_yaxes(title_text="Chamfer Distance", row=1, col=col)
    fig_dual.update_xaxes(title_text="Epoch", row=1, col=col)

fig_dual.update_layout(
    title    = "PTv3 -- Training Loss (Linear & Log Scale)",
    template = "plotly_dark",
    height   = 450
)
fig_dual.show()
wandb.log({"loss_curve_dual_scale": wandb.Plotly(fig_dual)})

# ── Standalone log-scale curve ─────────────────────────────────────────
fig_log = go.Figure(go.Scatter(
    x=epochs_x, y=train_losses, mode="lines",
    line=dict(color="#FF7F0E", width=2), name="Train Loss"
))
fig_log.update_layout(
    title       = "PTv3 -- Training Loss (Log Scale)",
    xaxis_title = "Epoch",
    yaxis       = dict(type="log", title="Chamfer Distance (log scale)"),
    template    = "plotly_dark"
)
fig_log.show()
wandb.log({"loss_curve_log_scale": wandb.Plotly(fig_log)})

wandb.finish()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ghosalsohom2003 (ghosalsohom2003-own-use) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.
wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch   25 | Loss 0.022110 | LR 8.95e-04 | 7.1 min | ckpt saved
Epoch   50 | Loss 0.017684 | LR 5.41e-04 | 14.2 min | ckpt saved
Epoch   75 | Loss 0.013678 | LR 1.61e-04 | 21.4 min | ckpt saved
Epoch  100 | Loss 0.011832 | LR 0.00e+00 | 28.5 min | ckpt saved
Training done in 28.5 min.  Best loss: 0.011702


elapsed_min,▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇████
epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█
lr,▂▄▅███████▇▇▇▇▇▆▅▅▅▄▄▄▄▄▃▃▃▃▂▂▁▁▁▁▁▁▁▁▁▁
train_loss,█▅▅▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss_log,██▇▆▆▄▄▃▃▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
elapsed_min,28.49768
epoch,100
lr,0
train_loss,0.01183
train_loss_log,-1.92693


In [ ]:
# ─────────────────────────────────────────────
# PART 7: Evaluate & Visualize -- TRAIN SET
# Added: dual linear+log CD histogram
# ─────────────────────────────────────────────
wandb.init(project="maize-ptv3", name="ptv3-viz-train", resume="allow")

model = PTv3_Autoencoder(
    latent_dim = CONFIG["latent_dim"],
    num_points = CONFIG["num_points"],
    patch_size = CONFIG["patch_size"]
).to(device)
load_checkpoint(os.path.join(CKPT_DIR, "ptv3_best.pt"), model)
model.eval()

train_orig_list, train_recon_list = [], []
with torch.no_grad():
    for batch, _ in train_loader:
        batch = batch.to(device)
        recon, _ = model(batch)
        train_orig_list.append(batch.cpu().numpy())
        train_recon_list.append(recon.cpu().numpy())

train_orig  = np.concatenate(train_orig_list,  axis=0)
train_recon = np.concatenate(train_recon_list, axis=0)

train_cd = np.array([
    chamfer_distance(
        torch.tensor(train_orig[i]).unsqueeze(0),
        torch.tensor(train_recon[i]).unsqueeze(0)
    ).item()
    for i in range(len(train_orig))
])
print(f"Train -- Mean CD : {train_cd.mean():.6f} | Std : {train_cd.std():.6f}")

# Best 4 + Worst 4 3-D reconstructions
show_idx = list(np.argsort(train_cd)[:4]) + list(np.argsort(train_cd)[-4:])
labels   = [f"Best {i+1}" for i in range(4)] + [f"Worst {i+1}" for i in range(4)]

fig_train = make_subplots(
    rows=8, cols=2,
    specs=[[{"type":"scatter3d"}, {"type":"scatter3d"}]] * 8,
    subplot_titles=[
        t for lbl, idx in zip(labels, show_idx)
        for t in (f"{lbl} -- Original (CD={train_cd[idx]:.5f})",
                  f"{lbl} -- Reconstructed")
    ],
    vertical_spacing=0.02
)
for row, idx in enumerate(show_idx, start=1):
    o, r = train_orig[idx], train_recon[idx]
    fig_train.add_trace(go.Scatter3d(
        x=o[:,0], y=o[:,1], z=o[:,2], mode="markers",
        marker=dict(size=1.5, color=o[:,2], colorscale="Viridis", showscale=False),
        showlegend=False
    ), row=row, col=1)
    fig_train.add_trace(go.Scatter3d(
        x=r[:,0], y=r[:,1], z=r[:,2], mode="markers",
        marker=dict(size=1.5, color=r[:,2], colorscale="Plasma", showscale=False),
        showlegend=False
    ), row=row, col=2)

fig_train.update_layout(
    title="PTv3 Train Set -- Best 4 & Worst 4",
    template="plotly_dark", height=3600, margin=dict(l=10, r=10, t=60, b=10))
fig_train.show()
wandb.log({"train_reconstructions": wandb.Plotly(fig_train)})

# Dual CD histogram: linear + log10
fig_hist = make_subplots(rows=1, cols=2,
    subplot_titles=("CD Distribution (Linear)", "CD Distribution (Log10)"))
fig_hist.add_trace(go.Histogram(
    x=train_cd, nbinsx=40, name="Train CD", marker_color="#636EFA"), row=1, col=1)
fig_hist.add_trace(go.Histogram(
    x=np.log10(train_cd + 1e-10), nbinsx=40,
    name="Train log10(CD)", marker_color="#AB63FA"), row=1, col=2)
fig_hist.update_xaxes(title_text="Chamfer Distance", row=1, col=1)
fig_hist.update_xaxes(title_text="log10(CD)", row=1, col=2)
fig_hist.update_yaxes(title_text="Count", row=1, col=1)
fig_hist.update_layout(
    title="Train -- Chamfer Distance Distribution (Linear & Log)",
    template="plotly_dark", showlegend=False)
fig_hist.show()
wandb.log({"train_cd_hist_dual": wandb.Plotly(fig_hist)})

wandb.finish()


  Loaded -- epoch 99 | loss 0.011702
Train -- Mean CD : 0.012469 | Std : 0.003983


In [ ]:
# ─────────────────────────────────────────────
# PART 8: Evaluate & Visualize -- TEST SET
# Added: log-scale box plot, dual histograms,
#        latent PCA 2D & 3D, summary table
# ─────────────────────────────────────────────
wandb.init(project="maize-ptv3", name="ptv3-viz-test", resume="allow")

model = PTv3_Autoencoder(
    latent_dim = CONFIG["latent_dim"],
    num_points = CONFIG["num_points"],
    patch_size = CONFIG["patch_size"]
).to(device)
load_checkpoint(os.path.join(CKPT_DIR, "ptv3_best.pt"), model)
model.eval()

test_orig_list, test_recon_list = [], []
with torch.no_grad():
    for batch, _ in test_loader:
        batch = batch.to(device)
        recon, _ = model(batch)
        test_orig_list.append(batch.cpu().numpy())
        test_recon_list.append(recon.cpu().numpy())

test_orig  = np.concatenate(test_orig_list,  axis=0)
test_recon = np.concatenate(test_recon_list, axis=0)

test_cd = np.array([
    chamfer_distance(
        torch.tensor(test_orig[i]).unsqueeze(0),
        torch.tensor(test_recon[i]).unsqueeze(0)
    ).item()
    for i in range(len(test_orig))
])
print(f"Test  -- Mean CD : {test_cd.mean():.6f} | Std : {test_cd.std():.6f}")

# Best 4 + Worst 4 3-D reconstructions
show_idx = list(np.argsort(test_cd)[:4]) + list(np.argsort(test_cd)[-4:])
labels   = [f"Best {i+1}" for i in range(4)] + [f"Worst {i+1}" for i in range(4)]

fig_test = make_subplots(
    rows=8, cols=2,
    specs=[[{"type":"scatter3d"}, {"type":"scatter3d"}]] * 8,
    subplot_titles=[
        t for lbl, idx in zip(labels, show_idx)
        for t in (f"{lbl} -- Original (CD={test_cd[idx]:.5f})",
                  f"{lbl} -- Reconstructed")
    ],
    vertical_spacing=0.02
)
for row, idx in enumerate(show_idx, start=1):
    o, r = test_orig[idx], test_recon[idx]
    fig_test.add_trace(go.Scatter3d(
        x=o[:,0], y=o[:,1], z=o[:,2], mode="markers",
        marker=dict(size=1.5, color=o[:,2], colorscale="Viridis", showscale=False),
        showlegend=False
    ), row=row, col=1)
    fig_test.add_trace(go.Scatter3d(
        x=r[:,0], y=r[:,1], z=r[:,2], mode="markers",
        marker=dict(size=1.5, color=r[:,2], colorscale="Plasma", showscale=False),
        showlegend=False
    ), row=row, col=2)

fig_test.update_layout(
    title="PTv3 Test Set -- Best 4 & Worst 4",
    template="plotly_dark", height=3600, margin=dict(l=10, r=10, t=60, b=10))
fig_test.show()
wandb.log({"test_reconstructions": wandb.Plotly(fig_test)})

# Dual CD histogram
fig_hist_test = make_subplots(rows=1, cols=2,
    subplot_titles=("Test CD (Linear)", "Test CD (Log10)"))
fig_hist_test.add_trace(go.Histogram(
    x=test_cd, nbinsx=40, name="Test CD", marker_color="#EF553B"), row=1, col=1)
fig_hist_test.add_trace(go.Histogram(
    x=np.log10(test_cd + 1e-10), nbinsx=40,
    name="Test log10(CD)", marker_color="#FFA15A"), row=1, col=2)
fig_hist_test.update_xaxes(title_text="Chamfer Distance", row=1, col=1)
fig_hist_test.update_xaxes(title_text="log10(CD)", row=1, col=2)
fig_hist_test.update_layout(
    title="Test -- Chamfer Distance Distribution (Linear & Log)",
    template="plotly_dark", showlegend=False)
fig_hist_test.show()
wandb.log({"test_cd_hist_dual": wandb.Plotly(fig_hist_test)})

# Train vs Test box plot -- LOG SCALE y-axis
fig_box = go.Figure()
fig_box.add_trace(go.Box(y=train_cd, name="Train (seen)",
                         marker_color="#636EFA", boxmean=True))
fig_box.add_trace(go.Box(y=test_cd,  name="Test (unseen)",
                         marker_color="#EF553B", boxmean=True))
fig_box.update_layout(
    title    = "PTv3 -- Train vs Test Chamfer Distance (Log Scale)",
    yaxis    = dict(type="log", title="Chamfer Distance (log scale)"),
    template = "plotly_dark"
)
fig_box.show()
wandb.log({"train_vs_test_cd_log": wandb.Plotly(fig_box)})

# Latent PCA 2D & 3D
train_latents, test_latents = [], []
with torch.no_grad():
    for batch, _ in train_loader:
        _, z = model(batch.to(device))
        train_latents.append(z.cpu().numpy())
    for batch, _ in test_loader:
        _, z = model(batch.to(device))
        test_latents.append(z.cpu().numpy())

train_latents = np.concatenate(train_latents, axis=0)
test_latents  = np.concatenate(test_latents,  axis=0)
latents   = np.concatenate([train_latents, test_latents], axis=0)
split_lbl = (["Train"] * len(train_latents) + ["Test"] * len(test_latents))
print(f"Latents shape : {latents.shape}")
assert latents.shape[0] == len(split_lbl)

latent_2d = PCA(n_components=2).fit_transform(latents)
latent_3d = PCA(n_components=3).fit_transform(latents)

fig_pca2 = px.scatter(
    x=latent_2d[:,0], y=latent_2d[:,1], color=split_lbl,
    color_discrete_map={"Train":"#636EFA","Test":"#EF553B"},
    title="Latent Space PCA 2D -- Train vs Test",
    labels={"x":"PC1","y":"PC2","color":"Split"}, template="plotly_dark")
fig_pca2.show()
wandb.log({"latent_pca_2d": wandb.Plotly(fig_pca2)})

fig_pca3 = px.scatter_3d(
    x=latent_3d[:,0], y=latent_3d[:,1], z=latent_3d[:,2], color=split_lbl,
    color_discrete_map={"Train":"#636EFA","Test":"#EF553B"},
    title="Latent Space PCA 3D -- Train vs Test",
    labels={"x":"PC1","y":"PC2","z":"PC3","color":"Split"}, template="plotly_dark")
fig_pca3.update_traces(marker=dict(size=3))
fig_pca3.show()
wandb.log({"latent_pca_3d": wandb.Plotly(fig_pca3)})

# Summary table
wandb.log({
    "evaluation_summary": wandb.Table(
        columns=["Split","Samples","Mean CD","Std CD","Min CD","Max CD"],
        data=[
            ["Train", len(train_cd),
             round(float(train_cd.mean()),6), round(float(train_cd.std()),6),
             round(float(train_cd.min()),6),  round(float(train_cd.max()),6)],
            ["Test",  len(test_cd),
             round(float(test_cd.mean()),6),  round(float(test_cd.std()),6),
             round(float(test_cd.min()),6),   round(float(test_cd.max()),6)],
        ]
    )
})

wandb.finish()
print("All done.")


  Loaded -- epoch 99 | loss 0.011702
Test  -- Mean CD : 0.013228 | Std : 0.006658


Latents shape : (1041, 512)


All done.
